# Gemma 4 E4B document-type adapter: merge, GGUF export, Ollama check

Turns the fine-tuned LoRA ([apprentice-gemma4-e4b-lora-document-types](https://huggingface.co/singhabhishekkk/apprentice-gemma4-e4b-lora-document-types)) into one GGUF file that Ollama can serve, then proves it answers the document-type prompt locally. Nothing is claimed until the Ollama cell prints a real answer.

Runtime: **L4 recommended** (merging an 8B model in 16-bit needs more memory than a T4 has). Runtime -> Change runtime type -> L4 GPU. Run top to bottom.

In [ ]:
# 1. Hugging Face auto-login (never blocks Run all).
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
if token:
    from huggingface_hub import login
    login(token=token, add_to_git_credential=False)
    print("HF login: authenticated via Colab secret HF_TOKEN")
else:
    print("HF login: no HF_TOKEN secret, continuing unauthenticated (public repos)")
del token

In [ ]:
# 2. Install (Colab)
# unsloth 2026.6.9 pair: the combination proven on this task (see the training
# notebook's version ledger, 2026-07-12). gguf + sentencepiece for the export.
!pip install -q unsloth==2026.6.9 unsloth-zoo==2026.6.7 gguf sentencepiece
import unsloth, transformers
print(f"unsloth {unsloth.__version__}, transformers {transformers.__version__}")

In [ ]:
# 3. Load base + adapter in 16-bit and merge.
from unsloth import FastLanguageModel

ADAPTER = "singhabhishekkk/apprentice-gemma4-e4b-lora-document-types"
model, tokenizer = FastLanguageModel.from_pretrained(
    ADAPTER,          # unsloth resolves the base model from the adapter config
    max_seq_length=2048,
    load_in_4bit=False,
    dtype=None,
)
print("loaded adapter on its base in 16-bit")

In [ ]:
# 4. Export GGUF (merged weights, q4_k_m quantization: the Ollama default size).
# unsloth handles merge + llama.cpp conversion in one call. Prints progress.
model.save_pretrained_gguf("gguf_out", tokenizer, quantization_method="q4_k_m")
import glob, os
# unsloth writes the converted files into gguf_out_gguf/ (observed 2026-07-12),
# so glob both trees; ignore the BF16 intermediate and the vision mmproj file.
ggufs = [g for g in glob.glob("gguf_out*/**/*.gguf", recursive=True) if "mmproj" not in g]
q4 = [g for g in ggufs if "q4" in os.path.basename(g).lower()]
GGUF_PATH = q4[0] if q4 else ggufs[0]
for g in ggufs:
    print(f"  {g} ({os.path.getsize(g)/1e9:.2f} GB)")
print("using:", GGUF_PATH)

In [ ]:
# 5. Install Ollama inside Colab and serve the GGUF.
# Prefer unsloth's generated Modelfile: it carries Gemma's chat template and
# stop tokens, which a bare FROM line would miss.
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import glob, os, subprocess, time
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
generated = glob.glob("gguf_out*/Modelfile")
if generated:
    modelfile_path = generated[0]
else:
    open("Modelfile", "w").write(f"FROM ./{GGUF_PATH}\nPARAMETER temperature 0\n")
    modelfile_path = "Modelfile"
print("modelfile:", modelfile_path)
subprocess.run(["ollama", "create", "doc-type-preset", "-f", modelfile_path], check=True)
print("ollama model created: doc-type-preset")

In [ ]:
# 6. The proof: ask it to classify one held-out document through Ollama.
# Uses the same prompt shape as the benchmark. The printed answer is the claim gate:
# a real class name here = the GGUF works in Ollama. Garbage here = do not publish.
import json, urllib.request

CLASS_NAMES = ["ADVE", "Email", "Form", "Letter", "Memo", "News", "Note", "Report", "Resume", "Scientific"]
sample_text = (
    "MEMORANDUM  TO: All Department Heads  FROM: R. J. Reynolds  "
    "DATE: March 14, 1989  SUBJECT: Quarterly budget review meeting scheduled "
    "for Friday at 10am in conference room B. Please bring your cost projections."
)
prompt = (
    "I will provide you with the content and the title of a document.\n"
    "Your task is to select the most appropriate document type for the document "
    "from the list of available document types I will provide.\n"
    "Only select a document type from the provided list. Respond only with the "
    "selected document type name, without any additional information.\n\n"
    f"<available_document_types>\n{', '.join(CLASS_NAMES)}\n</available_document_types>\n\n"
    f"<content>\n{sample_text}\n</content>"
)
payload = json.dumps({"model": "doc-type-preset", "prompt": prompt, "stream": False}).encode()
req = urllib.request.Request("http://127.0.0.1:11434/api/generate", data=payload,
                             headers={"Content-Type": "application/json"})
with urllib.request.urlopen(req, timeout=300) as r:
    answer = json.load(r)["response"].strip()
print("Ollama answer:", repr(answer))
print("PASS: valid class name" if answer.strip().strip('"') in CLASS_NAMES else "FAIL: not a class name, do not publish")

In [ ]:
# 7. Optional: upload the GGUF to the adapter's HF repo (only after cell 6 prints PASS).
# Root of the repo, not a subfolder: `ollama run hf.co/<repo>` only finds
# GGUF files at the repo root.
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_file(path_or_fileobj=GGUF_PATH,
#                 path_in_repo="apprentice-gemma4-e4b-doc-types.Q4_K_M.gguf",
#                 repo_id="singhabhishekkk/apprentice-gemma4-e4b-lora-document-types")
# print("uploaded; test from any machine: ollama run hf.co/singhabhishekkk/apprentice-gemma4-e4b-lora-document-types")